# Week 38, Day 1: Project: library system

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w038d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


> **Key idea.** `shelf` is built in layers: values and errors at the bottom, then the domain (the rules), the store (the data), the service (the use cases), the interface (text in and out), and the composition root that wires them. Money is whole pence, dates are `date` objects, today comes from an injected function, and every refusal is a `LibraryError` with a message a person can act on.

### Exercise: Step 1: the error family

Write the exception family, each class with a docstring:

- `LibraryError(Exception)`, the base;
- `InvalidInput`, a `LibraryError` **and** a `ValueError` (it really is a bad value);
- `NotFound`, a `LibraryError` and a `LookupError`;
- `NotAllowed`, a `LibraryError`;
- `StorageError`, a `LibraryError`.

In [ ]:
class LibraryError(Exception):
    """Base class for the library's own errors."""

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_family():\n    \"the family\"\n    assert issubclass(LibraryError, Exception) and not issubclass(LibraryError, (ValueError, LookupError))\n    assert issubclass(InvalidInput, LibraryError) and issubclass(InvalidInput, ValueError)\n    assert issubclass(NotFound, LibraryError) and issubclass(NotFound, LookupError)\n    for cls in (NotAllowed, StorageError):\n        assert issubclass(cls, LibraryError) and not issubclass(cls, (ValueError, LookupError)), cls.__name__\n    for cls in (LibraryError, InvalidInput, NotFound, NotAllowed, StorageError):\n        assert cls.__doc__, f\"{cls.__name__} needs a docstring\"\n    assert str(NotFound(\"no book B9\")) == \"no book B9\"")

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('errors.py', 'w', encoding='utf-8') as f:
    f.write('"""Every error the library raises on purpose."""\n\n\nclass LibraryError(Exception):\n    """Base class for the library\'s own errors."""\n\n\nclass InvalidInput(LibraryError, ValueError):\n    """Something typed or loaded doesn\'t make sense."""\n\n\nclass NotFound(LibraryError, LookupError):\n    """There\'s no book or member with that id."""\n\n\nclass NotAllowed(LibraryError):\n    """The library\'s rules don\'t allow this right now."""\n\n\nclass StorageError(LibraryError):\n    """The library\'s data couldn\'t be read or saved."""\n')

### Exercise: Step 2: check an ISBN

Write `clean_isbn(text)`, which returns the ISBN as 13 digits with no hyphens or spaces. Strip the text, and remove hyphens and spaces inside it. Raise `InvalidInput`:

- `an ISBN has 13 digits: 'TEXT'` if what's left isn't exactly 13 ASCII digits (show the stripped text, with `repr`);
- `wrong check digit: 'TEXT'` if the weighted sum isn't a multiple of 10.

In [ ]:
from errors import InvalidInput

def clean_isbn(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def message(text):\n    try:\n        clean_isbn(text)\n    except InvalidInput as err:\n        return str(err)\n    raise AssertionError(f\"clean_isbn({text!r}) should raise InvalidInput\")\ndef test_good():\n    \"good ISBNs\"\n    for text, want in [(\"978-0-14-118776-1\", \"9780141187761\"), (\" 9780435950828 \", \"9780435950828\"), (\"978 1 84749 058 2\", \"9781847490582\")]:\n        assert clean_isbn(text) == want, f\"{text!r}\"\ndef test_bad():\n    \"bad ISBNs\"\n    assert message(\"123\") == \"an ISBN has 13 digits: '123'\"\n    assert message(\"  978-0-14-118776-X \") == \"an ISBN has 13 digits: '978-0-14-118776-X'\"\n    assert message(\"\") == \"an ISBN has 13 digits: ''\"\n    assert message(\"\uff19\uff17\uff180141187761\") == \"an ISBN has 13 digits: '\uff19\uff17\uff180141187761'\"\n    assert message(\"978-0-14-118776-2\") == \"wrong check digit: '978-0-14-118776-2'\"\n    assert message(\"9780141187671\") == \"wrong check digit: '9780141187671'\", \"two digits swapped\"")

### Exercise: Step 3: tidy text

Write `clean_text(text, what, longest)`, which returns the text with runs of spaces (and tabs and newlines) turned into single spaces and none at the ends. Raise `InvalidInput("WHAT is missing")` if nothing's left, or `InvalidInput("WHAT is longer than LONGEST characters")` if the tidied text is too long.

In [ ]:
from errors import InvalidInput

def clean_text(text, what, longest):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_text():\n    \"tidying\"\n    assert clean_text(\"  Things   Fall\\tApart \", \"title\", 100) == \"Things Fall Apart\"\n    assert clean_text(\"Ada\", \"name\", 3) == \"Ada\"\n    assert clean_text(\"a  \" * 30, \"author\", 60) == \" \".join([\"a\"] * 30), \"measure the length after tidying\"\n    for args, message in [((\"   \", \"title\", 100), \"title is missing\"), ((\"Ada Obi\", \"name\", 5), \"name is longer than 5 characters\")]:\n        try:\n            clean_text(*args)\n            assert False, f\"{args} should raise InvalidInput\"\n        except InvalidInput as err:\n            assert str(err) == message, f\"got {str(err)!r}\"")

### Exercise: Step 4: read an id

`KINDS` maps each id letter to a word. Write `parse_id(text, kind)`, where `kind` is `"B"` or `"M"`. It strips and upper-cases the text and returns it, if it's the letter followed by ASCII digits not starting with `0`. Otherwise it raises `InvalidInput("not a WORD id: 'TEXT'")`, with the word from `KINDS` and the stripped text.

In [ ]:
from errors import InvalidInput

KINDS = {"B": "book", "M": "member"}

def parse_id(text, kind):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ids():\n    \"ids\"\n    assert parse_id(\" b1 \", \"B\") == \"B1\" and parse_id(\"M12\", \"M\") == \"M12\" and parse_id(\"m305\", \"M\") == \"M305\"\n    for text, kind, shown in [(\"B1\", \"M\", \"'B1'\"), (\"B\", \"B\", \"'B'\"), (\"B0\", \"B\", \"'B0'\"), (\"B01\", \"B\", \"'B01'\"), (\" BX1 \", \"B\", \"'BX1'\"),\n                              (\"1\", \"M\", \"'1'\"), (\"\", \"B\", \"''\"), (\"B-3\", \"B\", \"'B-3'\"), (\"B\u0661\", \"B\", \"'B\u0661'\")]:\n        try:\n            parse_id(text, kind)\n            assert False, f\"parse_id({text!r}, {kind!r}) should raise InvalidInput\"\n        except InvalidInput as err:\n            assert str(err) == f\"not a {KINDS[kind]} id: {shown}\", f\"got {str(err)!r}\"")

### Exercise: Step 5: read an amount

`format_pence` is given. Write `parse_pence(text)`, which strips the text, allows one `£` at the start, and accepts whole pounds with an optional `.` and one or two digits: `"0.6"` is `60`, `"£1.20"` is `120`, `"3"` is `300`. Anything else raises `InvalidInput("not an amount: 'TEXT'")` with the stripped text. (Zero is allowed here; tomorrow's rules decide what a payment of £0 means.)

In [ ]:
from errors import InvalidInput

def format_pence(pence):
    """60 as '£0.60', -350 as '-£3.50'."""
    sign = "-" if pence < 0 else ""
    pounds, rest = divmod(abs(pence), 100)
    return f"{sign}£{pounds:,}.{rest:02d}"

def parse_pence(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_amounts():\n    \"amounts\"\n    for text, want in [(\"0.6\", 60), (\"\u00a31.20\", 120), (\" 3 \", 300), (\"\u00a30.05\", 5), (\"0\", 0), (\"12.5\", 1250)]:\n        assert parse_pence(text) == want, f\"{text!r}: got {parse_pence(text)!r}\"\n    for text in [\"\", \"\u00a3\", \"1.234\", \"1.\", \".5\", \"-1\", \"\u00a3\u00a31\", \"1,000\", \"ten\", \"1 0\", \"\u00a3 1\"]:\n        try:\n            parse_pence(text)\n            assert False, f\"{text!r} should raise InvalidInput\"\n        except InvalidInput as err:\n            assert str(err) == f\"not an amount: {text.strip()!r}\"\n    assert format_pence(parse_pence(\"\u00a31.20\")) == \"\u00a31.20\"")

### Exercise: Step 6: check everything

Write `check_all(**checks)`. Each keyword argument is a function of no arguments. Call each in order; collect the results in a dictionary by name. If any raised `InvalidInput`, raise **one** `InvalidInput` whose message is all their messages joined by `"; "`, in order. Any other exception is a bug and must go straight through.

In [ ]:
from errors import InvalidInput

def check_all(**checks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def bad(message):\n    def check():\n        raise InvalidInput(message)\n    return check\ndef test_all_good():\n    \"all good\"\n    assert check_all(a=lambda: 1, b=lambda: \"x\") == {\"a\": 1, \"b\": \"x\"} and check_all() == {}\ndef test_problems():\n    \"every problem\"\n    try:\n        check_all(isbn=bad(\"an ISBN has 13 digits: '1'\"), title=lambda: \"ok\", author=bad(\"author is missing\"))\n        assert False, \"should raise InvalidInput\"\n    except InvalidInput as err:\n        assert str(err) == \"an ISBN has 13 digits: '1'; author is missing\", f\"got {str(err)!r}\"\ndef test_bugs():\n    \"bugs go through\"\n    try:\n        check_all(a=bad(\"x\"), b=lambda: 1 / 0)\n        assert False\n    except ZeroDivisionError:\n        pass")

In [ ]:
from errors import InvalidInput, LibraryError

KINDS = {"B": "book", "M": "member"}

def clean_isbn(text):
    shown = text.strip()
    digits = shown.replace("-", "").replace(" ", "")
    if len(digits) != 13 or not (digits.isascii() and digits.isdigit()):
        raise InvalidInput(f"an ISBN has 13 digits: {shown!r}")
    total = sum(int(d) * (3 if i % 2 else 1) for i, d in enumerate(digits))
    if total % 10 != 0:
        raise InvalidInput(f"wrong check digit: {shown!r}")
    return digits

def clean_text(text, what, longest):
    cleaned = " ".join(text.split())
    if not cleaned:
        raise InvalidInput(f"{what} is missing")
    if len(cleaned) > longest:
        raise InvalidInput(f"{what} is longer than {longest} characters")
    return cleaned

def check_all(**checks):
    results, problems = {}, []
    for name, check in checks.items():
        try:
            results[name] = check()
        except InvalidInput as err:
            problems.append(str(err))
    if problems:
        raise InvalidInput("; ".join(problems))
    return results

def check_book(isbn, title, author):
    return check_all(isbn=lambda: clean_isbn(isbn),
                     title=lambda: clean_text(title, "title", 100),
                     author=lambda: clean_text(author, "author", 60))

for args in [("978-0-14-118776-1", " Things  Fall Apart", "Chinua Achebe"),
             ("978-0-14-118776-2", "", "Chinua Achebe"),
             ("9781847490582", "Half of a Yellow Sun", "Chimamanda Ngozi Adichie")]:
    try:
        print(check_book(*args))
    except LibraryError as err:
        print(f"error ({type(err).__name__}): {err}")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## The brief

A community library lends a few hundred books to its members. Until now, loans have been written in an exercise book, which makes it hard to see who has what, what's overdue, and who's waiting for the most popular titles. You'll write `shelf`, a program the volunteers type commands into:

```text
> add 978-0-14-118776-1 "Things Fall Apart" "Chinua Achebe"
added B1: Things Fall Apart by Chinua Achebe
> join "Ada Obi"
joined M1: Ada Obi
> join "Bo Lee"
joined M2: Bo Lee
> lend B1 M1
B1 lent to Ada Obi, due 2026-11-23
> reserve B1 M2
Bo Lee is number 1 in the queue for B1
> return B1
B1 returned 3 days late: fine £0.60; keep it for Bo Lee
> lend B1 M1
error: B1 is being kept for someone else
> pay M1 0.60
Ada Obi now owes £0.00
```

What it must do, by the end of the week:

1. **Add books** (an ISBN, a title and an author) and **members** (a name). Each gets an id: `B1`, `B2`, ... and `M1`, `M2`, ..., never reused.
2. **Lend** a book to a member for 21 days, and take it **back**, charging a fine for each day late.
3. Let members **reserve** a book that's out. When it comes back, it's kept for the first person in the queue, and only they can borrow it.
4. Take **payments** towards fines.
5. List what's **overdue**, and **find** books by words in the title or author.
6. **Save** everything to a JSON file after every change, safely.
7. Turn every problem into a clear message, log what it does, and be tested throughout.

The library's rules:

- A member may have at most **3** books at once.
- A member can't borrow while they have an **overdue** book, or while they owe **£5** or more.
- A late book costs **20p a day**, up to **£5** for one loan.
- An **ISBN** is the 13-digit number on the back of a book, often written with hyphens; its last digit is a check digit (Step 2). A **title** is at most 100 characters, an **author** or a member's **name** at most 60, with spaces tidied.
- Ids are typed in any case and with spaces around them: ` b1 ` means `B1`.

## A plan

This program has more rules than `spend`, and Phase 3 gave you better tools for them. Following Week 36, it's split into **layers**, each in its own module, each depending only on the ones below:

| Layer | Module | Its job | Day |
|---|---|---|---|
| | `errors.py` | the exception family | 1 (today) |
| | `values.py` | read and check typed values; show money | 1 |
| domain | `domain.py` | `Book`, `Member`, `Loan` and the lending rules | 2 |
| infrastructure | `store.py` | keep the data in memory or in a JSON file | 3 |
| application | `service.py` | the use cases: add, lend, return, reserve, pay, find | 4 |
| presentation | `cli.py` | turn typed lines into calls, and results into text | 4 |
| composition root | `app.py` | build the program from settings, and run it | 4 |
| | `tests/` | pytest tests for every layer | 5 |

Some decisions, made now so they don't have to be made in a hurry later:

- **Classes for the things**, as dataclasses (Week 32): a `Book` knows who has it and who's waiting; a `Loan` knows its dates and can work out its fine. The rules live in the domain, which never prints, reads input or touches files.
- **Money in whole pence**, as in `spend`: integers add exactly.
- **Dates as `datetime.date` objects** inside the program, turned into ISO text only when saved. The service gets "today" from a function passed in (Week 36's injected clock), so tests can choose the day.
- **One exception family.** Everything the program refuses on purpose is a `LibraryError`, so the interface can catch that one class and show its message, and anything else (a bug) still crashes with a traceback (Week 37).
- **Generous at the edge, strict inside** (Week 37 Day 5): `values.py` accepts `" b1 "` and `978-0-14-118776-1`; everything after it works with `"B1"` and `"9780141187761"`.

In this week's exercises, each editor starts with the code from the steps before it, already working, and the finished modules from earlier days are in files on the page.

## Step 1: the errors

Start with the errors, because every other module raises them. Callers need to tell four situations apart: bad input (fix what you typed), something that doesn't exist (check the id), something the rules forbid (wait, or pay), and data that can't be read or saved (call for help).

From here on, the family is in `errors.py`, and the other modules import from it:

## Step 2: ISBNs

An ISBN-13 has 13 digits, and the last one is a **check digit**, chosen so that a simple sum comes out as a multiple of 10. Multiply the digits by 1, 3, 1, 3, ... in turn and add them up:

```text
digits:   9  7  8  0  1  4  1  1  8  7  7  6  1
weights:  1  3  1  3  1  3  1  3  1  3  1  3  1
products: 9 21  8  0  1 12  1  3  8 21  7 18  1    total 110: a multiple of 10, so it's valid
```

The check digit catches every mistake in a single digit, and most swaps of two neighbouring digits, which are the commonest typing errors. That's why it's worth checking: a mistyped ISBN is refused at once, instead of creating a book nobody can find later.

## Step 3: tidy text

Titles, authors and names are typed by hand, with stray spaces. One function tidies them all, given what the text is (for the message) and how long it may be.

## Step 4: ids

Volunteers type ids quickly: `b1`, ` M12 `. The program accepts any case and spaces around, and turns them into one form, but refuses anything that isn't an id of the kind expected, including `B0` and `B01` (no id starts with 0), so `B01` and `B1` can't both seem to exist.

## Step 5: money

Fines are paid in pounds and pence, and stored as whole pence. `format_pence` is the same as `spend`'s; reading is simpler than `spend`'s, because payments are small: no commas.

## Step 6: every problem at once

When a volunteer adds a book with a mistyped ISBN **and** no title, they should hear about both at once, as in `spend`. `spend` wrote a `try` block for each field. With several use cases needing the same thing, it's worth one reusable helper: give it the checks as functions with no arguments (lambdas, Week 15), and it runs them all, collecting the problems.

## Checkpoint

Here's today's work collected into `values.py`, and used the way Day 4's service will use it, to check a new book:

The interface will catch `LibraryError` exactly like this, so every refusal reaches the volunteer as one clear line.

## Quiz

## Before you go

Make a folder called `shelf` on your own computer, start a Git repository in it, and put today's code in `errors.py` and `values.py` (`KINDS`, `clean_isbn`, `clean_text`, `parse_id`, `parse_pence`, `format_pence`, `check_all`). Commit. Tomorrow: the domain, with books, members, loans and the lending rules.